In [ ]:
# @title
print("Installation de librosa...")
!pip install librosa &> /dev/null

print("Importation des librairies...")

import numpy as np # Classique, utile pour manipuler des tableaux, des matrices
from matplotlib import pyplot as plt # Permet de générer des figures
import random # Permet de générer des valeurs aléatoires (entier entre 0 et n, flottant entre 0 et 1, liste de nombres aléatoires...)
import itertools # Permet d'agir de manière itératives sur des données structurées

import tensorflow as tf # Librairie de référence dans la programmation d'IA avec python

import keras # Librairie de référence dans la programmation de réseau de neurones avec python
from keras.models import Sequential # Permet d'arranger les différentes couches du réseau de neurones
from keras.models import Model  # Permet de construire un modèle de réseau de neurones, de l'entraîner et de l'exploiter
from keras.layers import Dense # Pour ajout d'une couche dense
from keras.layers import Flatten # Pour ajout d'une couche de flatten
from keras.layers import Dropout # Pour ajout d'une couche de dropout (utilité : permet desfois d'éviter l'overfitting).
from keras.layers import Activation # Importe les différences fonction d'activation ('relu','sigmoid','sofmax',...)
from keras.layers import Conv2D # Pour ajout d'une couche de convolution 2D
from keras.layers import MaxPooling2D #Pour ajout d'une couche de MaxPooling 2D
from keras.utils import to_categorical # A vous de trouver !

from sklearn.metrics import confusion_matrix # Permet d'obtenir la matrice de confusion du modèle en croisant les labels avec les prédictions du réseau

import os # Utile pour naviguer à travers les différents dossier
from google.colab import drive # Utile pour connecter le notebook à un drive google
from pathlib import Path # Facilite la manipulation des chemins vers des fichiers

from IPython.display import Audio # Permet la lecture et l'enregistrement de fichier audio directement depuis le notebook

import librosa # Bibliothèque utilisée pour l'analyse et le traitement des données audio.

import glob # librairie spécialisée dans la recherche de chemin

from tqdm.auto import tqdm  # Fournit une barre de progression visuelle pour les boucles dans les notebooks Jupyter.

print("Installation et importation terminées !\n \n")

def connexion_drive(working_dir):
  print("Connexion au drive...")
  # point de montage de google drive dans google colab
  drive.mount('drive', force_remount=True)
  # création du dossier de travail si celui-ci n'existe pas
  if not os.path.exists(working_dir):
    os.mkdir(working_dir)
  print("Connecté au drive !")

def initialisation_dossier_melspec(data_dir):
  meldir = str(data_dir)+"/melspectrograms/"
  if not os.path.exists(meldir):
    os.mkdir(meldir)
  else:
    for elt in os.listdir(meldir):
      os.remove(meldir + elt)

def importation_des_donnees(data_dir):
  print("Importation des données...")
  # Si le dossier de données n'existe pas, on le créé
  freespoken_Dir = data_dir + "/free-spoken-digit-dataset"
  if not os.path.exists(data_dir):
    os.mkdir(data_dir)



Installation de librosa...
Importation des librairies...
Installation et importation terminées !
 



In [ ]:
# @title
################################################################################
# renseigner le chemin vers le dossier de travail
# dans lequel seront déposés les datas et les résultats : jeux de poids, ...
# Il reste juste à choisir le nom que vous voulez pour le working_dir (chemin)
# (i.e. renommer 2023-2024-S6-Projet-Audio)
#
# Utilisation du Drive sous Google Colab cliquer sur
# l'icone orange en haut à gauche permet d'accéder au Drive
# dans lequel vous pouvez créer des dossiers, stocker des données : dataset, poids d'un réseau entrainé, ...
################################################################################
working_dir = 'drive/MyDrive/PJT_IA_Musicale'
data_dir=working_dir+"/Datasets"

# Utilisation des fonctions d'initialisation
connexion_drive(working_dir)

Connexion au drive...
Mounted at drive
Connecté au drive !


In [ ]:
ClonageDataset=True
if ClonageDataset :
  importation_des_donnees(data_dir)
# Une fois importation_des_donnees() faite dans votre Drive, vous pouvez passer cette variable ClonageDataset à False.
# Cela evitera à chaque compilation de cloner à nouveau le dataset dans le Drive
if not os.path.exists(data_dir):
    os.makedirs(data_dir, exist_ok=True) # Create the parent directory if it doesn't exist
initialisation_dossier_melspec(data_dir)

Importation des données...


In [ ]:
 #title
def summarize_diagnostics(history,
                          directory,
                          filename):
    """
    pour visualiser l'entrainement du réseau,
    nous avons défini une procédure qui crée un graphique représentant l'évolution, au cours des époques :
      - Des valeurs de la fonction de perte (loss)
      - De la précision de classification (accuracy) pour les données d'entrainement et de test.

    Arguments
    ---------
    history:          historique de l'entraînement

    directory:        répertoire de sauvegarde des images

    filename:         Nom attribuée à l'image dans le dossier "directory"

    Usage
    -----
    summarize_diagnostics(history,
                          directory,
                          filename)



    """
    # plot loss
    plt.subplot(211)
    plt.title('Cross Entropy Loss')
    plt.plot(history.history['loss'], color='blue', label='train')
    plt.plot(history.history['val_loss'], color='orange', label='test')
    plt.legend()
    # plot accuracy
    plt.subplot(212)
    plt.title('Classification Accuracy')
    plt.plot(history.history['accuracy'], color='blue', label='train')
    plt.plot(history.history['val_accuracy'], color='orange', label='test')
    plt.legend()
    # save plot to file
    plt.savefig("{}/{}_plot.png".format(directory,filename))
    plt.show()
    plt.close()


def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=False):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')

    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

def occurencesclasses(label,numclasses):
  classes_list=np.arange(0,numclasses) # classes={0,1,...,N-1} pour un problème à N classes
  occurences=np.zeros(len(classes_list),int)
  for i in range(len(label)):
      classe=int(label[i])
      occurences[classe]=occurences[classe]+1
  return(occurences)

In [ ]:
#@title

# source ayant servi de base : https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be
# Permet d'enregistrer un fichier audio sur colab en utilisant le microphone par défaut de l'ordinateur.
from IPython.display import Javascript, Audio
from google.colab import output
from base64 import b64decode
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def button_record(filename='audio.wav',time=5):
  js = Javascript('''
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record  = time => new Promise(async resolve => {

    const div = document.createElement('div');
    const capture = document.createElement('button');
    capture.textContent = 'Capture';
    div.appendChild(capture);
    document.body.appendChild(div);

    stream = await navigator.mediaDevices.getUserMedia({audio: true});

    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)

    await new Promise((resolve) => capture.onclick = resolve);
    recorder.start()
    capture.textContent = "Recording"
    capture.style.color = "green"
    await sleep(time)
    recorder.onstop = async ()=>{
    blob = new Blob(chunks);
    text = await b2text(blob);
    resolve(text);
  }
    stream.getAudioTracks()[0].stop();
    recorder.stop();
    div.remove();
})
''')
  display(js)
  data = eval_js('record({})'.format(time*1000))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# Modifier le chemin du dossier record_rep ci-dessous comme vous le souhaitez.
# C'est l'endroit du drive où seront sauvegardés les enregistrements sous un nom de fichier .wav spécifié dans reord_path.

def record_audio(nom,dossier,duree=3):
  if not os.path.exists(dossier):
      os.mkdir(dossier)
  record_path= "{d}/{n}.wav".format(d=dossier,n=nom)
  # lancement de l'enregistrement
  button_record(filename=record_path,time=duree)
  # écouter l'enregistrement
  Audio(record_path)

def load_audio(nom,dossier):
  y, sr = librosa.load("{d}/{n}".format(d=dossier, n=nom), sr= 8000)
  #y est le signal sous forme d'array numpy
  # sr est le taux d'échantillonage (vaut 8khz  d'après les indications github)
  print(type(y),y, sr)
  Audio(y,rate=sr)
  return y, sr

In [ ]:
def plot_spec(S_dB):
  plt.figure(figsize=(10, 4))
  librosa.display.specshow(S_dB, x_axis='time',
                         y_axis='mel', sr=sr)
  plt.colorbar(format='%+2.0f dB')
  plt.title('Mel-frequency spectrogram')
  plt.tight_layout()
  plt.show()

def compute_spectrogram(path):
    "Calcule le melspectrogramme du fichier audio situé à path dans le repertoire de fichiers"
    file_name = os.path.basename(path)
    s , _ = librosa.load(path, sr= sr) # sr est le taux d'échantillonnage du signal.

    melspec = librosa.feature.melspectrogram(y=s, sr=sr, n_mels=n_mels,n_fft=1024)
    melspec_db = librosa.power_to_db(melspec, ref=np.max)
    # Nous rajoutons une dimension 1 à la fin pour forcer la représentation d'une image en tant que tenseur
    # (hauteur, largeur, nombre de canaux ici égal à 1)
    # pour compatibilité avec la première couche Conv2D du réseau de neurones
    melspec_db_image = melspec_db.reshape(melspec_db.shape[0],melspec_db.shape[1],1)
    return (path,melspec_db_image)

def get_duration(spec):
  "Renvoie la dimension correspondant à l'axe temporel d'un melspectrogramme"
  return spec.shape[1]

def get_longest_duration(list_path_spec):
  "Renvoie la dimension temporelle du melspectrogramme le plus long"
  list_time = [get_duration(spec) for path, spec in list_path_spec ]
  return max(list_time)

def save_melspec(path_melspec):
    """
    Permet d'enregistrer les spectrogrammes dans le dossier melspectrogram
    """
    path, melspec = path_melspec
    file_name = os.path.basename(path)
    melspec_path = os.path.join(data_dir,"melspectrograms", os.path.splitext(file_name)[0])
    np.save(melspec_path, melspec)
    #print("{} saved".format(melspec_path))

def save_fft(path_melspec):
  print(path_melspec)
  pass

In [ ]:
sr = 8000

# Génération d'un sinus à 512 HZ
y1 = 100*np.sin(2 * np.pi * 512 * np.linspace(0,2.5,int(8000*2.5)))
y2 = np.sin(2 * np.pi * 1024 * np.linspace(0,2.5,int(8000*2.5)))

y = np.concatenate([y1,y2])

sp = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128) # calcul du melspectrogramme
s_dB = librosa.power_to_db(sp, ref=np.max) #normalisation pour meilleure visualisation

print(type(s_dB))

plot_spec(s_dB)


<class 'numpy.ndarray'>


In [ ]:
# Fonction de conversion

working_dir = 'drive/MyDrive/PJT_IA_Musicale'
data_dir=working_dir+"/Datasets"

data_dir=working_dir+"/bpm-musique"
# Corrected: Use a wildcard to find all .mp3 files within the directory
list_of_files = glob.glob(os.path.join(data_dir, "*.mp3"))

i=0 # Application sur un des fichiers
print(list_of_files)
# affichage du nom d'un des fichers
# la nomenclature des fichiers du dataset est chiffreprononcé_nomduspeaker_occurence.wav où occurence={0,...,49}
print(list_of_files[i])

n_files = len(list_of_files) # nombres de fichiers audios (3000 pour vérification)
print(n_files)

x, sr = librosa.load(list_of_files[i], sr= 20000) #y est le signal sous forme d'array numpy, sr est le taux d'échantillonage (vaut 8khz  d'après les indications github)
print(type(x),np.shape(x),sr)

#lecture du fichier
Audio(x,rate=sr)

# création et affichage du mel-spectrogramme
sp = librosa.feature.melspectrogram(y=x, sr=sr, n_mels=128) # calcul du melspectrogramme
s_dB = librosa.power_to_db(sp, ref=np.max)
plot_spec(s_dB)

n_mels=128

musique/90 -  - Shackled Up.mp3', 'drive/MyDrive/PJT_IA_Musicale/bpm-musique/93 -  - Bring Me Down.mp3', 'drive/MyDrive/PJT_IA_Musicale/bpm-musique/90 -  - How To Cry.mp3', 'drive/MyDrive/PJT_IA_Musicale/bpm-musique/88 -  - Popcorn Salé.mp3', 'drive/MyDrive/PJT_IA_Musicale/bpm-musique/79 -  - What Was I Made For From The Motion Picture Barbie.mp3', 'drive/MyDrive/PJT_IA_Musicale/bpm-musique/86 -  - Les mots bleus.mp3', 'drive/MyDrive/PJT_IA_Musicale/bpm-musique/108_- _-_BOOM.mp3']
drive/MyDrive/PJT_IA_Musicale/bpm-musique/107 -  - Young  Stupid.mp3
112
<class 'numpy.ndarray'> (594254,) 20000


In [ ]:
# @title
################################################################################
# renseigner le chemin vers le dossier de travail
# dans lequel seront déposés les datas et les résultats : jeux de poids, ...
# Il reste juste à choisir le nom que vous voulez pour le working_dir (chemin)
# (i.e. renommer 2023-2024-S6-Projet-Audio)
#
# Utilisation du Drive sous Google Colab cliquer sur
# l'icone orange en haut à gauche permet d'accéder au Drive
# dans lequel vous pouvez créer des dossiers, stocker des données : dataset, poids d'un réseau entrainé, ...
################################################################################
working_dir = 'drive/MyDrive/PJT_IA_Musicale'
data_dir=working_dir+"/Datasets"

# Utilisation des fonctions d'initialisation
connexion_drive(working_dir)



list_of_files = glob.glob(data_dir+"/recordings/*") #permet d'obtenir le chemin relatif de tous les fichiers du dossier recordings


i=0
print(list_of_files)
# affichage du nom d'un des fichers
# la nomenclature des fichiers du dataset est chiffreprononcé_nomduspeaker_occurence.wav où occurence={0,...,49}
print(list_of_files[i])

n_files = len(list_of_files) # nombres de fichiers audios (3000 pour vérification)
print(n_files)

import librosa
import librosa.display
import matplotlib.pyplot as plt
import os

audio_file = list_of_files[i]
# audio_path = os.path.join(data_dir, audio_file) # This line caused the error
audio_path = audio_file # Use the full path directly

# === 4. Charger le fichier audio ===
y, sr = librosa.load(audio_path, sr=None)
print(f"✅ Fichier chargé : {audio_file}")
print(f"Fréquence d’échantillonnage : {sr} Hz")
print(f"Durée : {librosa.get_duration(y=y, sr=sr):.2f} secondes")

# === 5. Détection du tempo et des battements ===
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
print(f"🎵 Tempo estimé : {tempo:.2f} BPM")

# === 6. Conversion en temps des battements ===
beat_times = librosa.frames_to_time(beat_frames, sr=sr)
print("Premiers battements détectés (s) :", beat_times[:10])

# === 7. Visualisation ===
plt.figure(figsize=(12, 6))
librosa.display.waveshow(y, sr=sr, alpha=0.6)
plt.vlines(beat_times, -1, 1, color='r', alpha=0.8, linestyle='--', label='Battements détectés')
plt.title(f"Signal audio avec battements ({tempo:.2f} BPM)")
plt.xlabel("Temps (s)")
plt.ylabel("Amplitude")
plt.legend()
plt.show()

Connexion au drive...
Mounted at drive
Connecté au drive !
[]


**algorithme pour calculer le bpm**




In [ ]:
# @title
print("Installation de librosa...")
!pip install librosa &> /dev/null

print("Importation des librairies...")

import numpy as np # Classique, utile pour manipuler des tableaux, des matrices
from matplotlib import pyplot as plt # Permet de générer des figures
import random # Permet de générer des valeurs aléatoires (entier entre 0 et n, flottant entre 0 et 1, liste de nombres aléatoires...)
import itertools # Permet d'agir de manière itératives sur des données structurées

import tensorflow as tf # Librairie de référence dans la programmation d'IA avec python

import keras # Librairie de référence dans la programmation de réseau de neurones avec python
from keras.models import Sequential # Permet d'arranger les différentes couches du réseau de neurones
from keras.models import Model  # Permet de construire un modèle de réseau de neurones, de l'entraîner et de l'exploiter
from keras.layers import Dense # Pour ajout d'une couche dense
from keras.layers import Flatten # Pour ajout d'une couche de flatten
from keras.layers import Dropout # Pour ajout d'une couche de dropout (utilité : permet desfois d'éviter l'overfitting).
from keras.layers import Activation # Importe les différences fonction d'activation ('relu','sigmoid','sofmax',...)
from keras.layers import Conv2D # Pour ajout d'une couche de convolution 2D
from keras.layers import MaxPooling2D #Pour ajout d'une couche de MaxPooling 2D
from keras.utils import to_categorical # A vous de trouver !

from sklearn.metrics import confusion_matrix # Permet d'obtenir la matrice de confusion du modèle en croisant les labels avec les prédictions du réseau

import os # Utile pour naviguer à travers les différents dossier
from google.colab import drive # Utile pour connecter le notebook à un drive google
from pathlib import Path # Facilite la manipulation des chemins vers des fichiers

from IPython.display import Audio # Permet la lecture et l'enregistrement de fichier audio directement depuis le notebook

import librosa # Bibliothèque utilisée pour l'analyse et le traitement des données audio.

import glob # librairie spécialisée dans la recherche de chemin

from tqdm.auto import tqdm  # Fournit une barre de progression visuelle pour les boucles dans les notebooks Jupyter.

print("Installation et importation terminées !\n \n")

def connexion_drive(working_dir):
  print("Connexion au drive...")
  # point de montage de google drive dans google colab
  drive.mount('drive', force_remount=True)
  # création du dossier de travail si celui-ci n'existe pas
  if not os.path.exists(working_dir):
    os.mkdir(working_dir)
  print("Connecté au drive !")

def initialisation_dossier_melspec(data_dir):
  meldir = str(data_dir)+"/melspectrograms/"
  if not os.path.exists(meldir):
    os.mkdir(meldir)
  else:
    for elt in os.listdir(meldir):
      os.remove(meldir + elt)

def importation_des_donnees(data_dir):
  print("Importation des données...")
  # Si le dossier de données n'existe pas, on le créé
  freespoken_Dir = data_dir + "/free-spoken-digit-dataset"
  if not os.path.exists(data_dir):
    os.mkdir(data_dir)
  #téléchargement du jeu de données free-spoken-digit-dataset sur votre drive
  !git clone https://gitlab.ensam.eu/QCons/pjt1a_projetaudio $data_dir/free-spoken-digit-dataset
  !rm -rf $data_dir/free-spoken-digit-dataset/.git



**Conexion au drive et definition du fichier de travail**

In [ ]:
working_dir = 'drive/MyDrive/PJT IA Musicale'
data_dir=working_dir+"/bpm-musique"

# Utilisation des fonctions d'initialisation
connexion_drive(working_dir)


**Conversion des fichiers MP3 en WAV**

In [ ]:
from pydub import AudioSegment
import os # Import os module for path manipulation

# Fonction de conversion
data_dir=working_dir+"/musique_test"
# Corrected: Use a wildcard to find all .mp3 files within the directory
list_of_files = glob.glob(os.path.join(data_dir, "*.mp3"))


i=0 # Application sur un des fichiers
print(list_of_files)


def mp3_to_wav(input_file, output_file):
    # Charger le fichier MP3
    # mp3wave is the directory where WAV files will be saved
    mp3wave = str(working_dir)+"/Datasets/converted_wav/"
    if not os.path.exists(mp3wave):
      os.makedirs(mp3wave) # Use os.makedirs to create parent directories if they don't exist
    audio = AudioSegment.from_mp3(input_file)
    # Exporter en WAV
    audio.export(output_file, format="wav")
    print(f"Conversion terminée : {output_file}")

# Only proceed if there are MP3 files found
if list_of_files:
    for i in range(len(list_of_files)):
        input_mp3 = list_of_files[i]   # Remplacez par votre fichier mp3
        # Corrected: Construct the output WAV path using os.path.join and os.path.splitext
        file_name_without_ext = os.path.splitext(os.path.basename(input_mp3))[0]
        output_wav = os.path.join(working_dir, "Datasets", "converted_wav", file_name_without_ext + ".wav")
        mp3_to_wav(input_mp3,output_wav)
    print("Toutes les conversions sont terminées !")
else:
    print("Aucun fichier .mp3 trouvé dans le répertoire spécifié.")



Tout premier programme de calcul de BPM, avec une tentative de détetction de changement de bpm anormale mais  avec en abssise des "échantillon de temps"


In [ ]:
def calcul_bpm(audio_path):
    bpm_max=160 #parametrer par rapport au compte humain
    # 1 charger le signal
    y, sr = librosa.load(audio_path, sr=None, mono=True) #permet de charger le fichier audio et de recupérer la fréqunce d'echantillonage
    print(sr)

    #2. Calculer l’enveloppe d’énergie, Utilisation du signal absolu et d’un filtre passe-bas pour lisser
    fc=200 #fréquence de coupure à choisir
    Wn=fc*2/sr #coefficient normalisé à mettre dans la fonction signal.butter pour définir la frequence de coupure de notre filtre passe bas
    energy = np.abs(y)  #ici l'energie est la valeur absolu du signal d'entrée, signal d'entrée qui est l’amplitude du son dans le temps
    b, a = signal.butter(4, Wn, 'low') #permet créer un filtre passe-bas de  d’ordre 4 avec une coupure à la fréquence de coupure choisi en Hz
    envelope = signal.filtfilt(b, a, energy)

    # === 3. Détection des onsets ===
    peaks, _ = signal.find_peaks(envelope, height=np.mean(envelope)*1.5, distance=sr*0.2) #cette fonction permet de detcter les maxiaux locaux d'une fonction (pics), pour la fonction enveloppe,
    #"height=np.mean(envelope)*1.5" permet de garder seulement  les pics significativement forts, pour éviter les petits bruits.
    #distance=sr*0.2 permet d'imposer un intervalle minimum entre deux pics (ici 0,2 s  soit maximum 5 pics/s environ  300 BPM).

    if len(peaks) < 2: #si pas assez de pics detécté on stoppe le programme
        print("⚠️ Trop peu de battements détectés pour estimer un BPM.")
        return None

    # === 4. Calcul des intervalles entre pics ===
    intervals = np.diff(peaks) / sr  # permet de calculer la distance entre deux pics en valeur d'echantillons, la division par sr permet de donner le resultat en secondes

    # 4. Calcul des intervalles et BPM
    intervals = np.diff(peaks) / sr
    bpm = 60.0 / intervals  # BPM instantanés

    bpm_variable = []  #tentavtive V1 de detection d'un bpm variable au cours du temps
    for i in range(1, len(bpm)):
        if bpm[i-1] == 0:
            continue
        ratio = bpm[i] / bpm[i-1]
        if ratio > 3 or ratio < 1/3:
            bpm_variable.append((i, bpm[i]))

    # 5. Affichage
    plt.figure(figsize=(12, 5))
    plt.plot(envelope, label="Enveloppe du signal")
    plt.plot(peaks, envelope[peaks], 'rx', label="Battements détectés")

    if not bpm_variable:
        bpm_median = np.median(bpm)
        if bpm_median > bpm_max:
            bpm_median = bpm_median / 2
        plt.title(f"Tempo estimé : {bpm_median:.1f} BPM")
        plt.xlabel("echantillon temporel paramétré par rapport à la freq echantilllonage")
        plt.legend()
        plt.show()
        return bpm_median
    else:
        plt.title("Variations anormales détectées dans le tempo")
        plt.xlabel("Échantillons")
        plt.legend()
        plt.show()
        print("⚠️ Variations anormales détectées :")
        for x, val in bpm_variable:
            print(f" - Pic {x} → {val:.1f} BPM")
        return bpm_variable


**programme de calcul de BPM, mais avec en abssise des le temps en seconde**


In [ ]:
# @title
import numpy as np
import scipy.signal as signal
import librosa
import matplotlib.pyplot as plt

data_dir=working_dir+"/Datasets"+"/converted_wav" #permet d'obtenir le chemin relatif de tous les fichiers du dossier recordings

import glob
list_of_files = glob.glob(data_dir + "/*.wav")

i=0
print(list_of_files[0])
# affichage du nom d'un des fichers


#avant de commencer une fichier .wav est fichier regroupant des  valeurs numériques représentant l’amplitude du son dans le temps)
#mais egalement  quelques données comme le taux d’échantillonnage, nombre de canaux et profondeur en bits

def calcul_bpm_viz_bas(audio_path):
    # === 1. Charger le signal ===
    y, sr = librosa.load(audio_path, sr=None, mono=True)


    fc=100#fréquence de coupure à choisir
    Wn=fc*2/sr #coefficient normalisé à mettre dans la fonction signal.butter pour définir la frequence de coupure de notre filtre passe bas
    # === 2. Calculer l’enveloppe d’énergie ===
    energy = np.abs(y)
    b, a = signal.butter(4, Wn, 'low')
    envelope = signal.filtfilt(b, a, energy)

    # === 3. Détection des pics (battements) ===
    peaks, _ = signal.find_peaks(envelope, height=np.mean(envelope)*1.5, distance=sr*0.2)

    if len(peaks) < 2:
        print("⚠️ Trop peu de battements détectés pour estimer un BPM.")
        return None

    # === 4. Calcul BPM ===
    intervals = np.diff(peaks) / sr
    mean_interval = np.median(intervals)
    bpm = 60.0 / mean_interval
    if bpm > 140:
        bpm = bpm / 2

    # === 5. Création de l’axe temporel en secondes ===
    time = np.arange(len(y)) / sr
    time_peaks = peaks / sr

    # === 6. Visualisation combinée ===
    plt.figure(figsize=(14, 6))

    # Signal original (bleu clair)
    plt.plot(time, y, color='lightblue', alpha=0.6, label="Signal audio original")

    # Enveloppe (orange)
    plt.plot(time, envelope, color='orange', linewidth=2, label="Enveloppe d’énergie")

    # Pics détectés (rouge)
    plt.plot(time_peaks, envelope[peaks], 'rx', label="Battements détectés")

    # Mise en forme du graphe
    plt.title(f"Détection des battements – Tempo estimé : {bpm:.1f} BPM")
    plt.xlabel("Temps (secondes)")
    plt.ylabel("Amplitude / Énergie")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"🎵 BPM estimé : {bpm:.1f}")
    return bpm


def calcul_bpm_viz_haut(audio_path):
    # === 1. Charger le signal ===
    y, sr = librosa.load(audio_path, sr=None, mono=True)


    fc=5000#fréquence de coupure à choisir
    Wn=fc*2/sr #coefficient normalisé à mettre dans la fonction signal.butter pour définir la frequence de coupure de notre filtre passe bas
    # === 2. Calculer l’enveloppe d’énergie ===
    energy = np.abs(y)
    b, a = signal.butter(4, Wn, 'high')
    envelope = signal.filtfilt(b, a, energy)

    # === 3. Détection des pics (battements) ===
    peaks, _ = signal.find_peaks(envelope, height=np.mean(envelope)*1.5, distance=sr*0.2)

    if len(peaks) < 2:
        print("⚠️ Trop peu de battements détectés pour estimer un BPM.")
        return None

    # === 4. Calcul BPM ===
    intervals = np.diff(peaks) / sr
    mean_interval = np.median(intervals)
    bpm = 60.0 / mean_interval
    if bpm > 140:
        bpm = bpm / 2

    # === 5. Création de l’axe temporel en secondes ===
    time = np.arange(len(y)) / sr
    time_peaks = peaks / sr

    # === 6. Visualisation combinée ===
    plt.figure(figsize=(14, 6))

    # Signal original (bleu clair)
    plt.plot(time, y, color='lightblue', alpha=0.6, label="Signal audio original")

    # Enveloppe (orange)
    plt.plot(time, envelope, color='orange', linewidth=2, label="Enveloppe d’énergie")

    # Pics détectés (rouge)
    plt.plot(time_peaks, envelope[peaks], 'rx', label="Battements détectés")

    # Mise en forme du graphe
    plt.title(f"Détection des battements – Tempo estimé : {bpm:.1f} BPM")
    plt.xlabel("Temps (secondes)")
    plt.ylabel("Amplitude / Énergie")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f"🎵 BPM estimé : {bpm:.1f}")
    return bpm




    # === 6. Affichage ===
    plt.figure(figsize=(12, 5))
    plt.plot(envelope, label="Enveloppe du signal")
    plt.plot(peaks, envelope[peaks], 'rx', label="Battements détectés")
    plt.title(f"Tempo estimé : {bpm:.1f} BPM")
    plt.xlabel("temps ")
    plt.legend()
    plt.show()
    #return bpm

# Exemple d’utilisation

audio_path = list_of_files[0]
bpm = calcul_bpm_viz_haut(audio_path)
print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])

print(f"🎵 BPM estimé haut : {bpm:.1f}")

compare=0
precision= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm)*100
print("écart est de :", precision)
bpm = calcul_bpm_viz_bas(audio_path)
print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])

print(f"🎵 BPM estimé bas : {bpm:.1f}")

compare=0
precision= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm)*100
print("écart est de :", precision)

test sur toutes les musiques du fichiers


In [ ]:

compare=0
pourcentage_erreur_maxi=15
for i in range(len(list_of_files)):
  audio_path = list_of_files[i]
  print(audio_path)
  bpm_bas = calcul_bpm_viz_bas(audio_path)
  precision_basse= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm_bas)*100
  bpm_haut = calcul_bpm_viz_haut(audio_path)
  precision_haute= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm_haut)*100
  if 100-pourcentage_erreur_maxi<precision_basse<100+pourcentage_erreur_maxi or 100-pourcentage_erreur_maxi<precision_haute<100+pourcentage_erreur_maxi :
    compare+=1
  else:
    print("''''''''''''''''''''''''")
    print("erreur")
    print(audio_path)
    print("le bpm calculé par le filtre passa bas est de:",bpm_bas)
    print("le bpm calculé par le filtre passa haut est de:",bpm_haut)
    print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
    print("écart est de :", precision)
    print("''''''''''''''''''''''''")

print("la précsions de notre algorithme avec un bpm de",pourcentage_erreur_maxi,"%","d'erreur est de",compare/len(list_of_files)*100)



Essai de combo  d'un filtre passe bas

In [ ]:
compare=0
pourcentage_erreur_maxi=15

for i in range(len(list_of_files)):
  audio_path = list_of_files[i]
  print(audio_path)
  bpm_bas = calcul_bpm_viz_bas(audio_path)
  precision_basse= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm_bas)*100
  bpm_haut = calcul_bpm_viz_haut(audio_path)
  precision_haute= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm_haut)*100
  if 100-pourcentage_erreur_maxi<precision_basse<100+pourcentage_erreur_maxi or 100-pourcentage_erreur_maxi<precision_haute<100+pourcentage_erreur_maxi :
    compare+=1
  else:
    bpm=(bpm_bas+bpm_haut)/2
    if 100-pourcentage_erreur_maxi<bpm<100+pourcentage_erreur_maxi:
      compare+=1
    else:
      print("''''''''''''''''''''''''")
      print("erreur")
      print(audio_path)
      print("le bpm calculé par le filtre passa bas est de:",bpm_bas)
      print("le bpm calculé par le filtre passa haut est de:",bpm_haut)
      print("le bpm combiné est de:",bpm)
      print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
      print("écart est de :", precision)
      print("''''''''''''''''''''''''")

print("la précsions de notre algorithme avec un bpm de",pourcentage_erreur_maxi,"%","d'erreur est de",compare/len(list_of_files)*100)


Essai d'un filtre coupe bande

In [ ]:
def calcul_bpm_viz_coupe_bande(audio_path, f1=100, f2=5000):
    # === 1. Charger le signal ===
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    Wn = [2*f1/sr, 2*f2/sr]  # normalisation
    b, a = signal.butter(4, Wn, btype='bandstop')

    energy = np.abs(y)
    envelope = signal.filtfilt(b, a, energy)

    peaks, _ = signal.find_peaks(envelope,height=np.mean(envelope)*1.5,distance=sr*0.2)

    if len(peaks) < 2:
        print("⚠️ Trop peu de battements détectés pour estimer un BPM.")
        return None

    # === 5. Calcul BPM ===
    intervals = np.diff(peaks) / sr
    mean_interval = np.median(intervals)
    bpm = 60.0 / mean_interval

    # Sécurité contre le BPM harmonique
    if bpm > 140:
        bpm = bpm / 2

    return bpm


In [ ]:
import numpy as np
import librosa
from scipy import signal
from scipy.signal import find_peaks

# ==============================================================
#  AIDE : filtre + scoring directement intégrés
# ==============================================================

def _apply_bandstop_envelope(y, sr, f1, f2, order=4):
    b, a = signal.butter(order, [2*f1/sr, 2*f2/sr], btype='bandstop')
    energy = np.abs(y)
    envelope = signal.filtfilt(b, a, energy)
    return envelope

def _envelope_score(envelope, sr):
    peaks, props = find_peaks(
        envelope,
        height=np.mean(envelope)*1.2,
        distance=int(sr * 0.2)
    )
    if len(peaks) < 2:
        return 0.0, peaks, props

    mean_peak_height = np.mean(props["peak_heights"])
    mean_env = np.mean(envelope)
    SNR_ratio = mean_peak_height / (mean_env + 1e-9)

    intervals = np.diff(peaks) / sr
    stability = 1.0 / (np.std(intervals) + 1e-6)

    score = 0.7 * SNR_ratio + 0.3 * stability
    return score, peaks, props


def _suggest_bandstop_candidates(y, sr, n_candidates=6, notch_width=40):
    S = np.abs(librosa.stft(y, n_fft=4096, hop_length=2048))
    mean_spectrum = np.mean(S, axis=1)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=4096)

    low_idx = np.searchsorted(freqs, 40)
    high_idx = np.searchsorted(freqs, 8000)

    spec_trim = mean_spectrum[low_idx:high_idx]
    freqs_trim = freqs[low_idx:high_idx]

    peak_idx, _ = find_peaks(
        spec_trim,
        height=np.mean(spec_trim)*1.5,
        distance=5
    )

    if len(peak_idx) == 0:
        return [(50, 50-20, 50+20), (60, 60-20, 60+20)]

    powers = spec_trim[peak_idx]
    order = np.argsort(powers)[::-1]
    chosen = peak_idx[order][:n_candidates]

    candidates = []
    for pi in chosen:
        f_center = freqs_trim[pi]
        f1 = max(20, f_center - notch_width)
        f2 = f_center + notch_width
        candidates.append((f_center, f1, f2))

    return candidates


# ==============================================================
#  FONCTION PRINCIPALE : BPM + sélection automatique coupe-bande
# ==============================================================

def calcul_bpm_viz_coupe_bande_auto(audio_path):
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    candidates = _suggest_bandstop_candidates(y, sr)

    best_score = -1
    best_env = None
    best_f1 = None
    best_f2 = None
    best_peaks = None

    for f_center, f1, f2 in candidates:
        env = _apply_bandstop_envelope(y, sr, f1, f2)
        score, peaks, props = _envelope_score(env, sr)

        if score > best_score:
            best_score = score
            best_env = env
            best_f1 = f1
            best_f2 = f2
            best_peaks = peaks

    if best_peaks is None or len(best_peaks) < 2:
        print("⚠️ Impossible d’estimer un BPM avec coupe-bande.")
        return None

    intervals = np.diff(best_peaks) / sr
    mean_interval = np.median(intervals)
    bpm = 60.0 / mean_interval

    if bpm > 140:
        bpm = bpm / 2

    print(f"🔧 Bande supprimée automatiquement : {best_f1:.1f} Hz – {best_f2:.1f} Hz")
    print(f"🎵 BPM estimé : {bpm:.1f}")

    return bpm


test sur toutes les musiques

In [ ]:

compare=0
pourcentage_erreur_maxi=10
for i in range(len(list_of_files)):
  audio_path = list_of_files[i]
  print(audio_path)
  bpm= calcul_bpm_viz_coupe_bande(audio_path)
  precision= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm)*100
  if 100-pourcentage_erreur_maxi<precision<100+pourcentage_erreur_maxi  :
    compare+=1
  else:
    print("''''''''''''''''''''''''")
    print("erreur")
    print(audio_path)
    print("le bpm calculé par le filtre coupe bande est de:",bpm)
    print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
    print("écart est de :", precision)
    print("''''''''''''''''''''''''")

print("la précsions de notre algorithme avec un bpm de",pourcentage_erreur_maxi,"%","d'erreur est de",compare/len(list_of_files)*100)



In [ ]:
def calcul_bpm_hpss(audio_path):
    # === 1. Charger le signal ===
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    # === 2. HPSS : séparation harmonique / percussive
    y_harm, y_perc = librosa.decompose.hpss(librosa.stft(y))
    y_perc = librosa.istft(y_perc)

    # === 3. Calculer l’enveloppe sur la partie percussive
    energy = np.abs(y_perc)

    # petit passe-bas pour lisser
    fc = 150
    Wn = 2*fc/sr
    b, a = signal.butter(4, Wn, btype='low')
    envelope = signal.filtfilt(b, a, energy)

    # === 4. Détection des pics ===
    peaks, props = signal.find_peaks(
        envelope,
        height=np.mean(envelope)*1.3,
        distance=int(sr * 0.2)
    )

    if len(peaks) < 2:
        print("⚠️ Trop peu de pics détectés.")
        return None

    # === 5. BPM ===
    intervals = np.diff(peaks) / sr
    mean_interval = np.median(intervals)
    bpm = 60.0 / mean_interval

    if bpm > 140:
        bpm = bpm / 2

    print(f"🎵 BPM estimé (HPSS) : {bpm:.1f}")
    return bpm


In [ ]:

compare=0
pourcentage_erreur_maxi=10
for i in range(len(list_of_files)):
  audio_path = list_of_files[i]
  print(audio_path)
  bpm= calcul_bpm_hpss(audio_path)
  precision= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm)*100
  if 100-pourcentage_erreur_maxi<precision<100+pourcentage_erreur_maxi  :
    compare+=1
  else:
    print("''''''''''''''''''''''''")
    print("erreur")
    print(audio_path)
    print("le bpm calculé par hpss est de:",bpm)
    print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
    print("écart est de :", precision)
    print("''''''''''''''''''''''''")

print("la précsions de notre algorithme avec un bpm de",pourcentage_erreur_maxi,"%","d'erreur est de",compare/len(list_of_files)*100)

In [ ]:
def calcul_bpm_ultra(audio_path):
    # === 1. Charger le signal ===
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    # === 2. HPSS : on garde la partie percussive
    S = librosa.stft(y)
    _, S_perc = librosa.decompose.hpss(S)
    y_perc = librosa.istft(S_perc)

    # === 3. Enveloppe du signal percussif + passe-bas
    energy = np.abs(y_perc)
    fc = 180
    Wn = 2 * fc / sr
    b, a = signal.butter(4, Wn, btype='low')
    envelope = signal.filtfilt(b, a, energy)

    # === 4. Détection des pics bruts ===
    peaks, props = signal.find_peaks(
        envelope,
        height=np.mean(envelope)*1.2,
        distance=int(sr * 0.2)
    )

    if len(peaks) < 2:
        print("⚠️ Pas assez de pics détectés.")
        return None

    # === 5. Nettoyage statistique des intervalles (type RANSAC) ===
    intervals = np.diff(peaks) / sr
    median_i = np.median(intervals)

    good_intervals = intervals[np.abs(intervals - median_i) < 0.2*median_i]
    if len(good_intervals) < 2:
        good_intervals = intervals  # fallback

    median_clean = np.median(good_intervals)
    bpm_raw = 60 / median_clean

    # === 6. Vérification par autocorrélation ===
    env_norm = envelope - np.mean(envelope)
    corr = np.correlate(env_norm, env_norm, mode='full')
    corr = corr[len(corr)//2:]  # positive lags

    lag_min = int(sr * 0.25)
    lag_max = int(sr * 2.0)

    corr_range = corr[lag_min:lag_max]
    best_lag = np.argmax(corr_range) + lag_min

    bpm_corr = 60 * sr / best_lag

    # === 7. Fusion intelligente ===
    bpm = (0.7*bpm_raw + 0.3*bpm_corr)

    # Correction harmonique
    if bpm > 180:
        bpm /= 2
    if bpm < 60:
        bpm *= 2

    print(f"🎵 BPM estimé (ULTRA) : {bpm:.2f}")
    return bpm


In [ ]:

compare=0
pourcentage_erreur_maxi=10
for i in range(len(list_of_files)):
  audio_path = list_of_files[i]
  print(audio_path)
  bpm= calcul_bpm_ultra(audio_path)
  precision= float(audio_path[53]+audio_path[54]+audio_path[55])/float(bpm)*100
  if 100-pourcentage_erreur_maxi<precision<100+pourcentage_erreur_maxi  :
    compare+=1
  else:
    print("''''''''''''''''''''''''")
    print("erreur")
    print(audio_path)
    print("le bpm calculé par hpss est de:",bpm)
    print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
    print("écart est de :", precision)
    print("''''''''''''''''''''''''")

print("la précsions de notre algorithme avec un bpm de",pourcentage_erreur_maxi,"%","d'erreur est de",compare/len(list_of_files)*100)

**programme d'arthur **


In [ ]:
import librosa
import librosa.display

audio_path = list_of_files[0]



# Charger un fichier audio
y, sr = librosa.load(audio_path)

# --- Détection du tempo (BPM) ---
tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
print(f"Tempo estimé : {tempo[0]:.2f} BPM")

# --- Détection de la tonalité ---
# Extraire les chromas
chroma = librosa.feature.chroma_cqt(y=y, sr=sr)

# Moyenne des chromas sur toute la chanson
chroma_mean = chroma.mean(axis=1)

# Liste des notes
notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

# Trouver la note dominante
dominant_note = notes[chroma_mean.argmax()]
print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
print(f"Tonalité dominante estimée : {dominant_note}")

**test avec toutes les musiques **

In [ ]:
compare=0
pourcentage_erreur_maxi=15

for i in range(len(list_of_files)):
  audio_path = list_of_files[i]
  print(audio_path)
  tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
  precision= float(audio_path[53]+audio_path[54]+audio_path[55])/float(tempo)*100
  if 100-pourcentage_erreur_maxi<precision<100+pourcentage_erreur_maxi :
    compare+=1
  else:
    print("''''''''''''''''''''''''")
    print("erreur")
    print(audio_path)
    print("le bpm calculé  est de:",tempo)
    print("bpm réél:",audio_path[53]+audio_path[54]+audio_path[55])
    print("écart est de :", precision)
    print("''''''''''''''''''''''''")

print("la précsions de notre algorithme avec un bpm de",pourcentage_erreur_maxi,"%","d'erreur est de",compare/len(list_of_files)*100)

Objectif combiner nos methodes de calcul de bpm par l'utulisation de 3 reseau de neuronnes

In [ ]:
import torch
import torch.nn as nn
import librosa
import numpy as np
from scipy import signal


data_dir=working_dir+"/Datasets"+"/converted_wav" #permet d'obtenir le chemin relatif de tous les fichiers du dossier recordings

# Populate list_of_files with the converted WAV files
import glob
list_of_files = glob.glob(data_dir + "/*.wav")

new one


In [ ]:
import torch
import torch.nn as nn
import librosa
import numpy as np
from scipy import signal
from sklearn.model_selection import train_test_split

# ---------------------------
# Chargement + padding
# ---------------------------
def load_and_pad(audio_paths, sr=44100):
    signals = []
    for path in audio_paths:
        y, _ = librosa.load(path, sr=sr, mono=True)
        signals.append(y)

    max_len = max(len(sig) for sig in signals)

    padded = []
    for sig in signals:
        if len(sig) < max_len:
            pad = np.zeros(max_len - len(sig))
            sig = np.concatenate([sig, pad])
        padded.append(sig)

    return np.stack(padded), sr

# ---------------------------
# Préprocessing — filtres
# ---------------------------
def preprocess_filters(y, sr, fc_low=150, fc_high=3000):
    energy = np.abs(y)

    Wn_low = 2 * fc_low / sr
    b_low, a_low = signal.butter(4, Wn_low, "low")
    env_low = signal.filtfilt(b_low, a_low, energy)

    Wn_high = 2 * fc_high / sr
    b_high, a_high = signal.butter(4, Wn_high, "high")
    env_high = signal.filtfilt(b_high, a_high, energy)

    harmonic, percussive = librosa.effects.hpss(y)
    env_hpss = np.abs(percussive)

    return env_low, env_high, env_hpss

# ---------------------------
# CNN (Low)
# ---------------------------
class BaseNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 8, 15, stride=2),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(8, 16, 15, stride=2),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )
        self.fc_out = nn.Linear(16 * 1000, 1)

    def forward(self, x):
        x = self.cnn(x)
        x = torch.flatten(x, 1)
        return self.fc_out(x)

# ---------------------------
# Réseau de fusion
# ---------------------------
class NetFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.fc(x)

# ---------------------------
# Forward complet
# ---------------------------
def forward_full(audio_paths, net_low, net_high, net_perc, net_fusion):
    signals, sr = load_and_pad(audio_paths)

    lows, highs, hpss = [], [], []
    for y in signals:
        e_low, e_high, e_hpss = preprocess_filters(y, sr)
        lows.append(e_low)
        highs.append(e_high)
        hpss.append(e_hpss)

    lows = torch.tensor(np.array(lows), dtype=torch.float32).unsqueeze(1)
    highs = torch.tensor(np.array(highs), dtype=torch.float32).unsqueeze(1)
    hpss = torch.tensor(np.array(hpss), dtype=torch.float32).unsqueeze(1)

    bpm_low = net_low(lows)
    bpm_high = net_high(highs)
    bpm_perc = net_perc(hpss)

    fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
    return net_fusion(fusion_in)

# ---------------------------
# Split homogène train/test
# ---------------------------
def split_dataset(X, y, test_size=0.2, seed=42):
    return train_test_split(X, y, test_size=test_size, random_state=seed, shuffle=True)

# ---------------------------
# Dataset PyTorch
# ---------------------------
# Fonction d'extraction du BPM à partir du chemin
# ===== Exemple : float(audio_path[53] + audio_path[54] + audio_path[55])
def extract_bpm_from_path(path):
    try:
        return float(path[53] + path[54] + path[55])
    except:
        raise ValueError(f"Impossible d'extraire le BPM depuis : {path}")

class BeatDataset(torch.utils.data.Dataset):
    def __init__(self, file_paths, labels, sr=44100):
        self.file_paths = file_paths
        # labels ignorés → extraction automatique
        self.labels = [extract_bpm_from_path(p) for p in file_paths]
        self.sr = sr

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        y, _ = librosa.load(path, sr=self.sr, mono=True)
        energy = np.abs(y)

        # Filtres
        Wn_low = 2 * 150 / self.sr
        b_low, a_low = signal.butter(4, Wn_low, "low")
        env_low = signal.filtfilt(b_low, a_low, energy)

        Wn_high = 2 * 3000 / self.sr
        b_high, a_high = signal.butter(4, Wn_high, "high")
        env_high = signal.filtfilt(b_high, a_high, energy)

        harmonic, percussive = librosa.effects.hpss(y)
        env_hpss = np.abs(percussive)

        return (
            torch.tensor(env_low, dtype=torch.float32).unsqueeze(0),
            torch.tensor(env_high, dtype=torch.float32).unsqueeze(0),
            torch.tensor(env_hpss, dtype=torch.float32).unsqueeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

# ---------------------------
# Entraînement
# ---------------------------
def train_model(train_ds, net_low, net_high, net_perc, net_fusion, epochs=10, lr=1e-3):
    loader = torch.utils.data.DataLoader(train_ds, batch_size=4, shuffle=True)
    optimizers = [
        torch.optim.Adam(net_low.parameters(), lr=lr),
        torch.optim.Adam(net_high.parameters(), lr=lr),
        torch.optim.Adam(net_perc.parameters(), lr=lr),
        torch.optim.Adam(net_fusion.parameters(), lr=lr)
    ]
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        total_loss = 0
        for low, high, perc, label in loader:
            bpm_low = net_low(low)
            bpm_high = net_high(high)
            bpm_perc = net_perc(perc)
            fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
            pred = net_fusion(fusion_in)

            loss = loss_fn(pred.squeeze(), label)

            for opt in optimizers:
                opt.zero_grad()
            loss.backward()
            for opt in optimizers:
                opt.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(loader):.4f}")

# ---------------------------
# Test + matrice d'erreur
# ---------------------------
def evaluate_model(test_ds, net_low, net_high, net_perc, net_fusion):
    loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False)
    preds = []
    labels = []

    for low, high, perc, label in loader:
        bpm_low = net_low(low)
        bpm_high = net_high(high)
        bpm_perc = net_perc(perc)
        fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
        pred = net_fusion(fusion_in)

        preds.append(pred.item())
        labels.append(label.item())

    preds = np.array(preds)
    labels = np.array(labels)

    errors = preds - labels
    mse = np.mean(errors**2)
    mae = np.mean(np.abs(errors))

    print("Évaluation :")
    print(f"MSE : {mse:.2f}")
    print(f"MAE : {mae:.2f}")

    print("Matrice d'erreur :")
    print(errors)

    return preds, labels, errors

# ---------------------------
# Exemple d'utilisation avec barre d'avancement
# ---------------------------
from tqdm import tqdm

def train_model_with_progress(train_ds, net_low, net_high, net_perc, net_fusion, epochs=10, lr=1e-3):
    loader = torch.utils.data.DataLoader(train_ds, batch_size=4, shuffle=True)
    optimizers = [
        torch.optim.Adam(net_low.parameters(), lr=lr),
        torch.optim.Adam(net_high.parameters(), lr=lr),
        torch.optim.Adam(net_perc.parameters(), lr=lr),
        torch.optim.Adam(net_fusion.parameters(), lr=lr)
    ]
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        total_loss = 0
        pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for low, high, perc, label in pbar:
            bpm_low = net_low(low)
            bpm_high = net_high(high)
            bpm_perc = net_perc(perc)
            fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
            pred = net_fusion(fusion_in)

            loss = loss_fn(pred.squeeze(), label)

            for opt in optimizers:
                opt.zero_grad()
            loss.backward()
            for opt in optimizers:
                opt.step()

            total_loss += loss.item()
            pbar.set_postfix({"loss": total_loss/len(loader)})

train_ds = BeatDataset(list_of_files, labels=None)
net_low = BaseNet()
net_high = BaseNet()
net_perc = BaseNet()
net_fusion = NetFusion()
# train_model_with_progress(train_ds, net_low, net_high, net_perc, net_fusion, epochs=20, lr=1e-4)


In [ ]:
# Split (20% test)
train_files, test_files = train_test_split(list_of_files, test_size=0.2, shuffle=True)

# Datasets automatiques
train_ds = BeatDataset(train_files, labels=None)
test_ds  = BeatDataset(test_files, labels=None)


In [ ]:
net_low    = BaseNet()
net_high   = BaseNet()
net_perc   = BaseNet()
net_fusion = NetFusion()


In [ ]:
train_model_with_progress(
    train_ds,
    net_low,
    net_high,
    net_perc,
    net_fusion,
    epochs=20,
    lr=1e-4
)


In [ ]:
import torch
import torch.nn as nn
import librosa
import numpy as np
from scipy import signal
from sklearn.model_selection import train_test_split

# ---------------------------
# Chargement + padding
# ---------------------------
def load_and_pad(audio_paths, sr=44100):
    signals = []
    for path in audio_paths:
        y, _ = librosa.load(path, sr=sr, mono=True)
        signals.append(y)

    max_len = max(len(sig) for sig in signals) # This max_len is for the batch

    padded = []
    for sig in signals:
        if len(sig) < max_len:
            pad = np.zeros(max_len - len(sig))
            sig = np.concatenate([sig, pad])
        padded.append(sig)

    return np.stack(padded), sr

# ---------------------------
# Préprocessing — filtres
# ---------------------------
def preprocess_filters(y, sr, fc_low=150, fc_high=3000):
    energy = np.abs(y)

    Wn_low = 2 * fc_low / sr
    b_low, a_low = signal.butter(4, Wn_low, "low")
    env_low = signal.filtfilt(b_low, a_low, energy)

    Wn_high = 2 * fc_high / sr
    b_high, a_high = signal.butter(4, Wn_high, "high")
    env_high = signal.filtfilt(b_high, a_high, energy)

    harmonic, percussive = librosa.effects.hpss(y)
    env_hpss = np.abs(percussive)

    return env_low, env_high, env_hpss

# ---------------------------
# CNN (Low)
# ---------------------------
class BaseNet(nn.Module):
    def __init__(self, input_length):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 8, 15, stride=2),
            nn.ReLU(),
            nn.MaxPool1d(4),
            nn.Conv1d(8, 16, 15, stride=2),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )
        # Calculate the output size of the CNN dynamically
        # Ensure input_length is at least the receptive field for the dummy input
        min_input_length_for_cnn = 80 # Receptive field of the CNN based on kernel sizes and strides
        safe_input_length = max(input_length, min_input_length_for_cnn)
        self._set_fc_input_dim(safe_input_length)

    def _set_fc_input_dim(self, input_length):
        # Create a dummy tensor to pass through the CNN to determine output size
        dummy_input = torch.randn(1, 1, input_length)
        dummy_output = self.cnn(dummy_input)
        self.fc_input_dim = dummy_output.view(1, -1).size(1)
        self.fc_out = nn.Linear(self.fc_input_dim, 1)

    def forward(self, x):
        x = self.cnn(x)
        x = torch.flatten(x, 1)
        return self.fc_out(x)

# ---------------------------
# Réseau de fusion
# ---------------------------
class NetFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.fc(x)

# ---------------------------
# Forward complet
# ---------------------------
def forward_full(audio_paths, net_low, net_high, net_perc, net_fusion):
    signals, sr = load_and_pad(audio_paths)

    lows, highs, hpss = [], [], []
    for y in signals:
        e_low, e_high, e_hpss = preprocess_filters(y, sr)
        lows.append(e_low)
        highs.append(e_high)
        hpss.append(e_hpss)

    lows = torch.tensor(np.array(lows), dtype=torch.float32).unsqueeze(1)
    highs = torch.tensor(np.array(highs), dtype=torch.float32).unsqueeze(1)
    hpss = torch.tensor(np.array(hpss), dtype=torch.float32).unsqueeze(1)

    bpm_low = net_low(lows)
    bpm_high = net_high(highs)
    bpm_perc = net_perc(hpss)

    fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
    return net_fusion(fusion_in)

# ---------------------------
# Split homogène train/test
# ---------------------------
def split_dataset(X, y, test_size=0.2, seed=42):
    return train_test_split(X, y, test_size=test_size, random_state=seed, shuffle=True)

# ---------------------------
# Dataset PyTorch
# ---------------------------
class BeatDataset(torch.utils.data.Dataset):
    def __init__(self, file_paths, labels, sr=44100, max_target_len=None):
        self.file_paths = file_paths
        self.labels = labels
        self.sr = sr
        self.max_target_len = max_target_len

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        y, _ = librosa.load(path, sr=self.sr, mono=True)

        # Pad 'y' to max_target_len if provided
        if self.max_target_len is not None and len(y) < self.max_target_len:
            pad_amount = self.max_target_len - len(y)
            y = np.pad(y, (0, pad_amount), 'constant')
        elif self.max_target_len is not None and len(y) > self.max_target_len:
            y = y[:self.max_target_len] # Truncate if longer

        energy = np.abs(y)

        # Filtres
        Wn_low = 2 * 150 / self.sr
        b_low, a_low = signal.butter(4, Wn_low, "low")
        env_low = signal.filtfilt(b_low, a_low, energy)

        Wn_high = 2 * 3000 / self.sr
        b_high, a_high = signal.butter(4, Wn_high, "high")
        env_high = signal.filtfilt(b_high, a_high, energy)

        harmonic, percussive = librosa.effects.hpss(y)
        env_hpss = np.abs(percussive)

        return (
            torch.tensor(env_low.copy(), dtype=torch.float32).unsqueeze(0),
            torch.tensor(env_high.copy(), dtype=torch.float32).unsqueeze(0),
            torch.tensor(env_hpss.copy(), dtype=torch.float32).unsqueeze(0),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )

# ---------------------------
# Entraînement
# ---------------------------
def train_model(train_ds, net_low, net_high, net_perc, net_fusion, epochs=10, lr=1e-3):
    loader = torch.utils.data.DataLoader(train_ds, batch_size=4, shuffle=True)
    optimizers = [
        torch.optim.Adam(net_low.parameters(), lr=lr),
        torch.optim.Adam(net_high.parameters(), lr=lr),
        torch.optim.Adam(net_perc.parameters(), lr=lr),
        torch.optim.Adam(net_fusion.parameters(), lr=lr)
    ]
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        total_loss = 0
        for low, high, perc, label in loader:
            bpm_low = net_low(low)
            bpm_high = net_high(high)
            bpm_perc = net_perc(perc)
            fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
            pred = net_fusion(fusion_in)

            loss = loss_fn(pred.squeeze(), label)

            for opt in optimizers:
                opt.zero_grad()
            loss.backward()
            for opt in optimizers:
                opt.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} — Loss: {total_loss/len(loader):.4f}")

# ---------------------------
# Test + matrice d'erreur
# ---------------------------
def evaluate_model(test_ds, net_low, net_high, net_perc, net_fusion):
    loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False)
    preds = []
    labels = []

    for low, high, perc, label in loader:
        bpm_low = net_low(low)
        bpm_high = net_high(high)
        bpm_perc = net_perc(perc)
        fusion_in = torch.cat([bpm_low, bpm_high, bpm_perc], dim=1)
        pred = net_fusion(fusion_in)

        preds.append(pred.item())
        labels.append(label.item())

    preds = np.array(preds)
    labels = np.array(labels)

    errors = preds - labels
    mse = np.mean(errors**2)
    mae = np.mean(np.abs(errors))

    print("Évaluation :")
    print(f"MSE : {mse:.2f}")
    print(f"MAE : {mae:.2f}")

    print("Matrice d'erreur :")
    print(errors)

    return preds, labels, errors

In [ ]:
# 1) Determine maximum audio length for BaseNet initialization
# Load all files to find the max length for padding
max_audio_len = 0
for path in list_of_files:
    y, _ = librosa.load(path, sr=44100, mono=True)
    if len(y) > max_audio_len:
        max_audio_len = len(y)

# Define a minimum reasonable length for the CNN (e.g., receptive field)
# The CNN architecture requires at least 133 for all layers to produce positive output dimensions.
min_cnn_input_length = 200 # Increased for safety based on CNN architecture
max_audio_len = max(max_audio_len, min_cnn_input_length)

# Generate labels from file paths
labels = [extract_bpm_from_path(path) for path in list_of_files]

# 2) Split train/test
train_files, test_files, train_labels, test_labels = split_dataset(list_of_files, labels)

# 3) Dataset
train_ds = BeatDataset(train_files, train_labels, max_target_len=max_audio_len)
test_ds  = BeatDataset(test_files, test_labels, max_target_len=max_audio_len)

# 4) Instanciation modèles, passing the maximum input length
net_low   = BaseNet(max_audio_len)
net_high  = BaseNet(max_audio_len)
net_perc  = BaseNet(max_audio_len)
net_fusion = NetFusion()

# 5) Entraînement
train_model(train_ds, net_low, net_high, net_perc, net_fusion, epochs=10, lr=1e-4)

# 6) Test
preds, labs, errors = evaluate_model(test_ds, net_low, net_high, net_perc, net_fusion)

In [ ]:
import librosa
import librosa.display
import numpy as np
import scipy.signal
import matplotlib.pyplot as plt





import os
import librosa


# --- 1.  Chargement ---
#audio_path = 'drive/MyDrive/PJT IA Musicale/Datasets/converted_wav/BLACKPINK_-_Kill_This_Love_MV.wav'
audio_path = 'drive/MyDrive/PJT IA Musicale/Datasets/converted_wav/The_Beatles_-_The_Beatles_-_Let_It_Be_Official_Music_Video_Remastered_2015.wav'
y, sr = librosa.load(audio_path)


hop_length = 8192

# --- 2. Extraction des caractéristiques  ---
chromagram = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)

# --- 3. Création de la Matrice d'Auto-Similarité (SSM) ---
# On calcule la distance entre chaque trame chromatique
ssm = librosa.segment.recurrence_matrix(chromagram, mode='affinity', metric='cosine', sym=True)

# Lissage optionnel pour réduire le bruit (filtre médian)
ssm_smoothed = scipy.ndimage.median_filter(ssm, size=(3, 3))

# --- 4. Fonction de Nouveauté
kernel_size = 25
kernel = np.kron(np.array([[1, -1], [-1, 1]]), np.ones((kernel_size, kernel_size)))
gauss_window = scipy.signal.windows.gaussian(kernel_size * 2, std=kernel_size/2)
kernel = kernel * np.outer(gauss_window, gauss_window)

novelty_curve = np.zeros(ssm_smoothed.shape[0])
for i in range(kernel_size, ssm_smoothed.shape[0] - kernel_size):
    local_block = ssm_smoothed[i-kernel_size:i+kernel_size, i-kernel_size:i+kernel_size]
    novelty_curve[i] = np.sum(local_block * kernel)

novelty_curve = np.maximum(novelty_curve, 0.0)
novelty_curve = novelty_curve / np.max(novelty_curve)

# --- 5. Détection des pics (Filtrage Topologique Strict) ---
# distance=40 équivaut à un minimum d'environ 15 secondes entre chaque phrase(pas toujpours vrai en musique )
peaks, _ = scipy.signal.find_peaks(
    novelty_curve,
    prominence=0.08,
    distance=40
)

boundaries_sec = librosa.frames_to_time(peaks, sr=sr, hop_length=hop_length)
print(f"Frontières détectées (en secondes) : {np.round(boundaries_sec, 2)}")

# --- 6. Visualisation ---
plt.figure(figsize=(12, 8))

# Affichage de la SSM
plt.subplot(2, 1, 1)
librosa.display.specshow(ssm_smoothed, y_axis='time', x_axis='time', hop_length=hop_length, sr=sr)
plt.title('Matrice d\'Auto-Similarité (SSM)')
plt.colorbar()

# Affichage de la courbe de nouveauté et des pics
plt.subplot(2, 1, 2)
time_axis = librosa.frames_to_time(np.arange(len(novelty_curve)), sr=sr, hop_length=hop_length)
plt.plot(time_axis, novelty_curve, label='Novelty Function')
plt.vlines(boundaries_sec, 0, 1, color='r', alpha=0.8, linestyle='--', label='Phrases (Peaks)')
plt.title('Détection des frontières de phrases')
plt.xlabel('Temps (s)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import os
import librosa
import librosa.display
import numpy as np
import scipy.signal
import matplotlib.pyplot as plt

# --- 1. Chargement --

y, sr = librosa.load( 'drive/MyDrive/PJT IA Musicale/Datasets/converted_wav/BLACKPINK_-_Kill_This_Love_MV.wav', sr=22050)
hop_length = 8192 # Résolution d'environ 0.37 secondes par trame

# --- 2. Matrice Hybride (Le secret pour la musique moderne) ---
chromagram = librosa.feature.chroma_cqt(y=y, sr=sr, hop_length=hop_length)
mfcc = librosa.feature.mfcc(y=y, sr=sr, hop_length=hop_length, n_mfcc=13)

# Normalisation et fusion
features = np.vstack([librosa.util.normalize(chromagram), librosa.util.normalize(mfcc)])

# --- 3. SSM ---
ssm = librosa.segment.recurrence_matrix(features, mode='affinity', metric='cosine', sym=True)
ssm_smoothed = scipy.ndimage.median_filter(ssm, size=(7, 7)) # Lissage extrême (7x7)

# --- 4. Fonction de Nouveauté (Échelle MACRO) ---
# Un noyau colossal de 80 trames (environ 30 secondes de champ de vision)
kernel_size = 80
kernel = np.kron(np.array([[1, -1], [-1, 1]]), np.ones((kernel_size, kernel_size)))
gauss_window = scipy.signal.windows.gaussian(kernel_size * 2, std=kernel_size/2)
kernel = kernel * np.outer(gauss_window, gauss_window)

novelty_curve = np.zeros(ssm_smoothed.shape[0])
for i in range(kernel_size, ssm_smoothed.shape[0] - kernel_size):
    local_block = ssm_smoothed[i-kernel_size:i+kernel_size, i-kernel_size:i+kernel_size]
    novelty_curve[i] = np.sum(local_block * kernel)

novelty_curve = np.maximum(novelty_curve, 0.0)
novelty_curve = novelty_curve / np.max(novelty_curve)

# --- 5. Filtrage Topologique ---
# distance=80 force un minimum de ~30 secondes entre chaque grande partie
peaks, _ = scipy.signal.find_peaks(
    novelty_curve,
    prominence=0.10, # Seuil très exigeant, on ne veut que les montagnes
    distance=80
)

boundaries_sec = librosa.frames_to_time(peaks, sr=sr, hop_length=hop_length)
print(f"Grandes frontières structurelles (en secondes) : {np.round(boundaries_sec, 2)}")

# --- 6. Visualisation ---
plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
librosa.display.specshow(ssm_smoothed, y_axis='time', x_axis='time', hop_length=hop_length, sr=sr)
plt.title('SSM Hybride - Lissage Macro')
plt.colorbar()

plt.subplot(2, 1, 2)
time_axis = librosa.frames_to_time(np.arange(len(novelty_curve)), sr=sr, hop_length=hop_length)
plt.plot(time_axis, novelty_curve, label='Macro Novelty')
plt.vlines(boundaries_sec, 0, 1, color='r', alpha=0.8, linestyle='--', linewidth=2)
plt.title('Détection des blocs principaux (Intro / Couplets / Refrains)')
plt.xlabel('Temps (s)')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def evaluate_boundaries(true_boundaries, predicted_boundaries, tolerance=3.0):
    """
    Évalue les performances d'un algorithme de détection de structure musicale.
    """
    true_bnd = np.array(true_boundaries)
    pred_bnd = np.array(predicted_boundaries)

    matched_predictions = set()
    tp = 0 # Vrais Positifs (True Positives)

    # Pour chaque vraie frontière, on cherche la prédiction valide la plus proche
    for t_val in true_bnd:
        # Calcul des distances absolues avec toutes les prédictions
        distances = np.abs(pred_bnd - t_val)

        # Identifier les prédictions dans la fenêtre de tolérance
        # ET qui n'ont pas encore été assignées à une autre vraie frontière
        valid_indices = [
            j for j, d in enumerate(distances)
            if d <= tolerance and j not in matched_predictions
        ]

        if valid_indices:
            # Trouver l'index de la prédiction la plus proche parmi les valides
            best_match_idx = valid_indices[np.argmin(distances[valid_indices])]

            tp += 1
            matched_predictions.add(best_match_idx) # On verrouille cette prédiction

    # Faux Positifs (FP) : Prédictions qui n'ont matché aucune vraie frontière
    fp = len(pred_bnd) - tp

    # Faux Négatifs (FN) : Vraies frontières qui n'ont aucune prédiction proche
    fn = len(true_bnd) - tp

    # Calcul des métriques avec protection contre la division par zéro
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    if (precision + recall) > 0:
        f1_score = 2 * (precision * recall) / (precision + recall)
    else:
        f1_score = 0.0

    return {
        "Precision": np.round(precision, 3),
        "Rappel": np.round(recall, 3),
        "F1-Score": np.round(f1_score, 3),
        "Details": {"TP": tp, "FP": fp, "FN": fn}
    }

# --- TEST AVEC VOS DONNÉES DE "LET IT BE" ---
# Les timestamps réels (Vérité Terrain) que j'ai fournis précédemment
ground_truth = [0.0, 13.0, 36.0, 59.0, 77.0, 100.0, 118.0, 131.0, 168.0, 190.0, 213.0, 236.0]

# Les prédictions de votre dernier algorithme (issues de votre image)
predictions = [9.66, 30.46, 62.04, 76.9, 114.06, 154.18, 174.24, 201.36, 231.08]

resultats = evaluate_boundaries(ground_truth, predictions, tolerance=3.0)

print(f"Précision : {resultats['Precision']}")
print(f"Rappel    : {resultats['Rappel']}")
print(f"F1-Score  : {resultats['F1-Score']}")
print(f"Détails   : {resultats['Details']}")